In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.functions import col
from pyspark.sql.functions import monotonically_increasing_id, col

In [0]:
cie_df = spark.read.format("csv")\
    .option("header","true")\
    .option("inferschema","false")\
    .option("mode","PERMISSIVE")\
    .load("/Volumes/workspace/default/milestone/zomato.csv")
cie_df.show(10)

In [0]:
cie_df.printSchema()

In [0]:
cie_df = cie_df.dropDuplicates()
cie_df.show()

In [0]:
cie_df = cie_df.fillna("Unknown")

In [0]:
cie_df.show()

In [0]:
cie_df = cie_df.withColumnRenamed("name", "Name")

In [0]:
from pyspark.sql.functions import monotonically_increasing_id, col

restaurants = cie_df.select(
    monotonically_increasing_id().alias("restaurant_id"),
    col("Name"),
    col("location").alias("location_id"),
    col("cuisines").alias("cuisine_id"),
    col("rate").alias("rating_id"),
    col("approx_cost(for two people)").alias("cost"),
    col("rest_type").alias("type")
)

In [0]:
restaurants.show()

In [0]:
locations = cie_df.select(
    monotonically_increasing_id().alias("location_id"),
    col("location").alias("location_name")
).dropDuplicates()

In [0]:
locations.show()

In [0]:
cuisines = cie_df.select(
    monotonically_increasing_id().alias("cuisine_id"),
    col("cuisines").alias("cuisine_name")
).dropDuplicates()

In [0]:
cuisines .show()

In [0]:
ratings = cie_df.select(
    monotonically_increasing_id().alias("rating_id"),
    col("rate").alias("rating_value"),
    col("votes")
).dropDuplicates()

In [0]:
ratings.show()

In [0]:
restaurants.createOrReplaceTempView("restaurants")
locations.createOrReplaceTempView("locations")
cuisines.createOrReplaceTempView("cuisines")
ratings.createOrReplaceTempView("ratings")

In [0]:
restaurants.show(5)
locations.show(5)
cuisines.show(5)
ratings.show(5)

In [0]:
cie_df.createOrReplaceTempView("cie_df")

In [0]:
spark.sql("""
SELECT name, location
FROM cie_df
""").show()

In [0]:
spark.sql("""
SELECT name, rate
FROM cie_df
ORDER BY rate DESC
LIMIT 10
""").show()

In [0]:
spark.sql("""
SELECT location,
COUNT(*) AS restaurant_count
FROM cie_df
GROUP BY location
""").show()

In [0]:
spark.sql("""
SELECT cuisines,
AVG(CAST(SPLIT(rate, '/')[0] AS DOUBLE)) AS average_rating
FROM cie_df
WHERE rate LIKE '%/%'
GROUP BY cuisines
ORDER BY average_rating DESC
""").show(truncate=False)

In [0]:
spark.sql("""
SELECT location,
COUNT(*) AS restaurant_count
FROM cie_df
GROUP BY location
HAVING COUNT(*) > 10
""").show()

In [0]:
spark.sql("""
SELECT location,
cuisines,
COUNT(*) AS total
FROM cie_df
GROUP BY location, cuisines
ORDER BY total DESC
""").show()

In [0]:
spark.sql("""
SELECT location,
SUM(try_cast(votes AS DOUBLE)) AS total_votes
FROM cie_df
GROUP BY location
""").show()

In [0]:
spark.sql("""
SELECT location,
cuisines,
COUNT(*) AS total
FROM cie_df
GROUP BY location, cuisines
ORDER BY total DESC
""").show()

In [0]:
spark.sql("""
SELECT Name, rate, votes
FROM cie_df
WHERE SPLIT(rate,'/')[0] > '4'
AND votes > '500'
""").show()

In [0]:
spark.sql("""
SELECT cuisines,
AVG(CAST(REPLACE(`approx_cost(for two people)`, ',', '') AS DOUBLE)) AS average_cost
FROM cie_df
WHERE `approx_cost(for two people)` <> 'Unknown'
GROUP BY cuisines
ORDER BY average_cost DESC
""").show(truncate=False)